In [1]:
import pandas as pd

laptops = pd.read_csv("laptop_data_cleaned.csv")
laptops

,Brand,Model,Processor,RAM_GB,Storage_GB,GPU,Screen_Size,Weight_kg,Launch_Year,Customer_Rating,Review_Count,Demand_Index,Competitor_Price,Price,Price_per_GB,Performance_Index,Discount_Factor,Launch_Category
0,MSI,Model_0,Ryzen 5,8,2048,Integrated,17,1,2019,4,4568,20,1372,300,0.146484,640,1072,Old
1,Apple,Model_1,Ryzen 9,8,128,Integrated,14,1,2023,3,3917,72,1784,300,2.343750,1728,1484,New
2,Asus,Model_2,i3,64,128,NVIDIA RTX 3050,17,2,2018,4,817,79,1383,300,2.343750,20224,1083,Old
3,MSI,Model_3,i7,8,512,NVIDIA RTX 3060,14,3,2022,3,196,53,1911,315,0.615234,1272,1596,Mid
4,Lenovo,Model_4,i5,32,1024,NVIDIA RTX 3080,17,2,2021,4,4837,23,751,317,0.309570,2944,434,Mid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,Asus,Model_494,i3,8,2048,NVIDIA RTX 3060,15,1,2022,3,400,69,1416,300,0.146484,1656,1116,Mid
457,MSI,Model_495,i3,27,801,NVIDIA RTX 3060,13,1,2018,3,2583,55,1683,404,0.504370,4455,1279,Old
458,Apple,Model_497,i9,8,2048,Integrated,17,2,2022,3,4847,98,2169,306,0.149414,2352,1863,Mid
459,Dell,Model_498,i7,16,128,NVIDIA RTX 3070,14,2,2021,4,4925,22,424,300,2.343750,1408,124,Mid


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score


In [7]:

X = laptops.drop(columns=["Price"])
y = laptops["Price"]

X = pd.get_dummies(X, drop_first=True)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

y_pred_lr = lin_reg.predict(X_test_scaled)


In [10]:
def evaluate_model(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name} → RMSE: {rmse:.2f}, R²: {r2:.3f}")
    
print("Linear Regression Performance:")
evaluate_model(y_test, y_pred_lr, "Linear Regression")


Linear Regression Performance:
Linear Regression → RMSE: 46.84, R²: 0.106


In [13]:

from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest Performance:")
evaluate_model(y_test, y_pred_rf, "Random Forest")

Random Forest Performance:
Random Forest → RMSE: 47.10, R²: 0.096


In [15]:
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost Performance:")
evaluate_model(y_test, y_pred_xgb, "XGBoost")


XGBoost Performance:
XGBoost → RMSE: 27.08, R²: 0.701


In [16]:
import joblib
joblib.dump(xgb, "xgboost_model.pkl")
joblib.dump(X.columns, "feature_columns.pkl")


['feature_columns.pkl']